# County weather point selection

This notebook calculates a population-weighted county centroid and matches it to the nearest WTK, BC-HRRR and NSRDB grid points. The worked example uses Arthur County, Nebraska (FIPS 31005); the national source files are too large for this compact demonstration.

**Inputs:** Two simplified Arthur County CSVs derived from 2020 Census population counts and TIGER block internal points, plus live HSDS grid metadata for the final matching step.

**Requirements:** The repository environment and a local HSDS service at `http://localhost:5101` with access to `nrel-pds-hsds` for Step 5. Steps 1–4 use bundled files.

**Outputs:** The Arthur County centroid and three selected grid IDs, displayed below. The [county weather notebook](county_hsds_download_and_ba_weather_aggregation.ipynb) demonstrates the next stage using the separately supplied [national county-point mapping](../../data_inputs/county_weather/county_centroid_regrid.csv).

Run the code cells from top to bottom. See the [setup instructions](../../README.md#quick-start).


**Saved-output provenance:** The local Arthur County calculations were checked in fresh kernels during an earlier review. The final grid-matching output is retained from an earlier successful live HSDS run; a local-only run does not refresh those lookups.


## Setup


In [1]:
from IPython.display import display

import pandas as pd
from rex import Resource
from scipy.spatial import cKDTree

## Step 1: Read Census block population

The national population source is the 2020 Census P.L. 94-171 block-level population counts. The compact `arthur_county_block_population_2020.csv` supplies the Arthur County example, using the total-population field `POP100`. Each block identifier is kept as a 15-digit text `GEOID20` so leading zeros are preserved before merging with TIGER coordinates.


In [2]:
block_population = pd.read_csv("../../data_inputs/county_weather/arthur_county_point_selection_example/arthur_county_block_population_2020.csv", dtype={"GEOID20": str})
print(f"Arthur County block rows: {len(block_population):,}")

Arthur County block rows: 46


## Step 2: Read TIGER block internal points

The national coordinate source is the 2020 TIGER/Line Block20 geodatabase. The compact `arthur_county_tiger_block_points_2020.csv` represents those block internal points for this example, keeping the same `GEOID20` plus the block internal-point latitude and longitude.


In [3]:
tiger_points = pd.read_csv("../../data_inputs/county_weather/arthur_county_point_selection_example/arthur_county_tiger_block_points_2020.csv", dtype={"GEOID20": str})
print(f"Arthur County TIGER rows: {len(tiger_points):,}")

Arthur County TIGER rows: 46


## Step 3: Merge blocks and extract county FIPS

The block population and TIGER internal-point tables join by `GEOID20`. For a 2020 Census block GEOID, the first 5 characters are the full county FIPS code: first 2 digits for state, followed by 3 digits for county within that state.


In [4]:
blocks = block_population.merge(tiger_points, on="GEOID20", validate="one_to_one")

# GEOID20 starts with state FIPS + county code; the first 5 characters are county FIPS.
blocks["county_fips"] = blocks["GEOID20"].str[:5]
print(f"Joined Arthur County block rows: {len(blocks):,}")
display(blocks.head(5))

Joined Arthur County block rows: 46


,GEOID20,POP100,INTPTLAT,INTPTLON,county_fips
0,310059583001000,6,41.730981,-101.438985,31005
1,310059583001001,4,41.703768,-101.476879,31005
2,310059583001002,8,41.734505,-101.577511,31005
3,310059583001003,2,41.704448,-101.636337,31005
4,310059583001004,2,41.731602,-101.694203,31005


## Step 4: Calculate the population-weighted county centroid

Population weighting gives more influence to blocks where more people live. The aim is to represent where county residents live when choosing weather inputs for load modeling.

For county `c`, the population-weighted centroid is:

`county_pop_lat_c = sum(POP100_b * INTPTLAT_b) / sum(POP100_b)`

`county_pop_lon_c = sum(POP100_b * INTPTLON_b) / sum(POP100_b)`

where `b` is a Census block in county `c`, `POP100_b` is block population from P.L. 94-171, and `INTPTLAT_b`/`INTPTLON_b` are TIGER block internal-point coordinates.


In [5]:
# Weight each block coordinate by that block's population.
blocks["POP100_times_INTPTLAT"] = blocks["POP100"] * blocks["INTPTLAT"]
blocks["POP100_times_INTPTLON"] = blocks["POP100"] * blocks["INTPTLON"]

# Sum population and weighted coordinates within each county FIPS.
# Keep the grouping visible before calculating its totals.
county_groups = blocks.groupby("county_fips", as_index=False)
county_centroid_example = county_groups.agg(population=("POP100", "sum"), sum_population_times_lat=("POP100_times_INTPTLAT", "sum"), sum_population_times_lon=("POP100_times_INTPTLON", "sum"))

# Divide weighted coordinate sums by total county population.
county_centroid_example["county_pop_lat"] = (county_centroid_example["sum_population_times_lat"] / county_centroid_example["population"])
county_centroid_example["county_pop_lon"] = (county_centroid_example["sum_population_times_lon"] / county_centroid_example["population"])
display(
    county_centroid_example.rename(
        columns={
            "county_fips": "County FIPS",
            "population": "Population",
            "sum_population_times_lat": "Population-weighted latitude sum",
            "sum_population_times_lon": "Population-weighted longitude sum",
            "county_pop_lat": "Centroid latitude",
            "county_pop_lon": "Centroid longitude",
        }
    )
    .style.hide(axis="index")
    .format(
        {
            "Population": "{:,.0f}",
            "Population-weighted latitude sum": "{:,.6f}",
            "Population-weighted longitude sum": "{:,.6f}",
            "Centroid latitude": "{:.6f}",
            "Centroid longitude": "{:.6f}",
        }
    )
)

County FIPS,Population,Population-weighted latitude sum,Population-weighted longitude sum,Centroid latitude,Centroid longitude
31005,434,"18,041.356747","-44,128.760654",41.569946,-101.679172


## Step 5: Match the county centroid to HSDS grid points

Nearest-grid matching uses HSDS grid metadata only: `gid`, grid latitude, and grid longitude. It does **not** download weather time series. The actual weather-variable reads happen in [county weather notebook](county_hsds_download_and_ba_weather_aggregation.ipynb).

Before running this step, start the local HSDS service at `http://localhost:5101` with access to the `nrel-pds-hsds` bucket.

This step reads metadata from one representative historical resource file for each weather source with `rex.Resource`, builds a `cKDTree` from the grid coordinates, and selects the nearest grid point for the example county centroid. Each weather source is matched separately because WTK, BC-HRRR, and NSRDB use different grids.


In [4]:
HSDS_ENDPOINT = "http://localhost:5101"
HSDS_API_KEY = None
HSDS_BUCKET = "nrel-pds-hsds"

# Use one representative resource year for each source's historical grid.
RESOURCE_PATHS = {
    "wtk": "/nrel/wtk/conus/wtk_conus_2013.h5",
    "bchrrr": "/nrel/wtk/bchrrr/v1.0.0/bchrrr_conus_2019.h5",
    "nsrdb": "/nrel/nsrdb/GOES/aggregated/v4.0.0/nsrdb_2019.h5",
}

centroid = county_centroid_example.iloc[0]
county_point = centroid[["county_pop_lat", "county_pop_lon"]].to_numpy(dtype=float)
selected_rows = []

for weather_source, resource_path in RESOURCE_PATHS.items():
    # 1. Read the latitude/longitude grid for this weather source.
    print(f"Looking up {weather_source} grid metadata through HSDS: {resource_path}")
    with Resource(
        resource_path,
        hsds=True,
        hsds_kwargs={
            "endpoint": HSDS_ENDPOINT,
            "api_key": HSDS_API_KEY,
            "bucket": HSDS_BUCKET,
        },
    ) as resource:
        grid_coordinates = resource.coordinates

    # 2. The coordinate row number is the HSDS grid ID.
    tree = cKDTree(grid_coordinates)
    _, selected_gid = tree.query(county_point)
    selected_gid = int(selected_gid)
    grid_lat, grid_lon = grid_coordinates[selected_gid]

    # 3. Store the selected grid ID and coordinates for this weather source.
    selected_rows.append(
        {
            "county_fips": centroid["county_fips"],
            "weather_source": weather_source,
            "resource_path": resource_path,
            "selected_gid": selected_gid,
            "county_pop_lat": float(centroid["county_pop_lat"]),
            "county_pop_lon": float(centroid["county_pop_lon"]),
            "grid_lat": float(grid_lat),
            "grid_lon": float(grid_lon),
        }
    )

nearest_grid_points = pd.DataFrame(selected_rows)
display(
    nearest_grid_points.rename(
        columns={
            "county_fips": "County FIPS",
            "weather_source": "Weather source",
            "resource_path": "HSDS resource",
            "selected_gid": "Grid ID",
            "county_pop_lat": "Centroid latitude",
            "county_pop_lon": "Centroid longitude",
            "grid_lat": "Grid latitude",
            "grid_lon": "Grid longitude",
        }
    )
    .style.hide(axis="index")
    .format(
        {
            "Grid ID": "{:.0f}",
            "Centroid latitude": "{:.6f}",
            "Centroid longitude": "{:.6f}",
            "Grid latitude": "{:.6f}",
            "Grid longitude": "{:.6f}",
        }
    )
)

Looking up wtk grid metadata through HSDS: /nrel/wtk/conus/wtk_conus_2013.h5


Looking up bchrrr grid metadata through HSDS: /nrel/wtk/bchrrr/v1.0.0/bchrrr_conus_2019.h5


Looking up nsrdb grid metadata through HSDS: /nrel/nsrdb/GOES/aggregated/v4.0.0/nsrdb_2019.h5


County FIPS,Weather source,HSDS resource,Grid ID,Centroid latitude,Centroid longitude,Grid latitude,Grid longitude
31005,wtk,/nrel/wtk/conus/wtk_conus_2013.h5,1014962,41.569946,-101.679172,41.575672,-101.680725
31005,bchrrr,/nrel/wtk/bchrrr/v1.0.0/bchrrr_conus_2019.h5,1014962,41.569946,-101.679172,41.575672,-101.680725
31005,nsrdb,/nrel/nsrdb/GOES/aggregated/v4.0.0/nsrdb_2019.h5,571348,41.569946,-101.679172,41.570000,-101.660004


## Interpretation

The centroid represents where residents live; it need not coincide with the county's geographic center. The three grid IDs refer to different weather grids and are used separately in the next stage. This Arthur County example demonstrates the selection method; it does not regenerate the supplied national mapping.
